<div style="display: flex; align-items: center; gap: 30px; margin: 18px 0 24px 0;">
  <img src="images/mcdc.png" alt="Metropolitan Chicago Data Science Corps logo" style="width: 150px; height: auto;">
  <img src="images/CivicLabLogo.jpeg" alt="The CivicLab logo" style="width: 230px; height: auto;">
</div>

## About This Lesson

This lesson is based on the Metropolitan Chicago Data Science Corps (MCDC) project [Tax Increment Financing (TIF) Illumination and Analysis](https://sites.northwestern.edu/mcdc/tax-increment-financing-illumination-and-analysis/), conducted in collaboration with [The CivicLab](https://www.civiclab.us/) for its [TIF Illumination Project](https://tifreports.com/). The project worked with annual Chicago TIF financial reports, converting information from PDF documents into structured data that could be analyzed and visualized.

Here we isolate one important part of that community-engaged workflow: programmatically obtaining a public PDF report and converting its financial information into an analysis-ready CSV. The [original project repository](https://github.com/philipayates/chicago2022TIF) contains the broader data-processing and visualization work.

Many public datasets are not distributed as convenient CSV files. Government agencies often publish information inside PDFs—documents designed for people to read, not for software to analyze. Data scientists may therefore need to retrieve a report, inspect its structure, extract and clean values, validate them, and create a structured file.

This lesson uses the **City of Chicago's Kinzie Industrial Corridor Tax Increment Financing (TIF) annual report** as its case study. We use the official FY 2019 report hosted by the Illinois Comptroller because it is available at a stable public URL and contains extractable text. The requested 2010 Kinzie figures appear later as historical reference values only; they are never used as extraction results.

> **Why this matters:** Data acquisition is part of data analysis. A polished CSV often hides substantial work needed to locate, interpret, and verify the original source.

## Learning objectives

By the end of this lesson, you should be able to:

1. Programmatically download a PDF from a public URL.
2. Distinguish an HTTP response from a file stored locally, inspect its status and headers, and verify that the resource is a PDF.
3. Open and inspect the structure of a PDF before attempting extraction.
4. Extract text and tabular financial information from a text-based PDF.
5. Clean semi-structured monetary data and convert it into an analysis-ready pandas DataFrame.
6. Validate automatically extracted data against the original public source.
7. Export the resulting structured data to CSV and verify the saved file.
8. Explain why PDF extraction is harder than reading CSV or HTML tables.
9. Explain how reproducible data acquisition and curation can make public information more usable for community-engaged data science.

## Setup

The installation command below is intentionally commented out. Uncomment it only if your environment does not already contain these packages.

In [1]:
# %pip install requests pdfplumber pandas

We use `Path` objects instead of manually assembled path strings. The `parents=True` option creates missing parent directories, and `exist_ok=True` makes the setup safe to run again.

In [2]:
from pathlib import Path

import pandas as pd
import pdfplumber
import requests

raw_dir = Path("data/raw")
processed_dir = Path("data/processed")
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

pdf_path = raw_dir / "kinzie_industrial_corridor_2019.pdf"
csv_path = processed_dir / "kinzie_2019_tif.csv"

## The Original Data Challenge

Chicago releases financial information about its TIF districts through annual reports. Although the reports are public, much of their information is stored inside individual PDF documents rather than in one analysis-ready table. For someone comparing districts or studying changes across years, manually opening reports and copying numbers would be time-consuming, difficult to reproduce, susceptible to transcription errors, and difficult to scale.

Programmatic acquisition and extraction can turn these reports into structured datasets:

<div style="font-family: monospace; line-height: 1.55; margin: 1em 0 1.4em 2em;">
City of Chicago TIF Reports<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;↓<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;PDF URLs<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;↓<br>
&nbsp;&nbsp;Download with Python<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;↓<br>
&nbsp;&nbsp;Extract PDF content<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;↓<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Clean values<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;↓<br>
&nbsp;&nbsp;pandas DataFrame<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;↓<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;CSV
</div>

The focus here is data acquisition and curation—not statistical analysis.

### MCDC Connection: Data Acquisition Is Data Science

Community-engaged projects often begin well before modeling or visualization. Public data may first need to be located, downloaded, interpreted, cleaned, documented, and validated so that later analysis rests on trustworthy evidence.

## Part 1 — Download a PDF from a URL

A **URL** identifies a resource on the web. `requests.get()` sends an HTTP GET request and returns a **Response object**. The object contains the server's status code, headers (metadata), and response body.

- `status_code` describes the outcome: codes in the 200s normally indicate success.
- `headers` contains metadata supplied by the server.
- `content` contains raw bytes, appropriate for a binary file such as a PDF.
- `text` decodes bytes into characters and is appropriate for textual web content—not for saving a PDF.

In [3]:
report_url = (
    "https://files.illinoiscomptroller.gov/LocGovTIF/FY2019/01662030/"
    "19TIF01662030Kinzie_InCorridor_1.pdf"
)

response = requests.get(report_url, timeout=30)
print("Status code:", response.status_code)
response.raise_for_status()

Status code: 200


`raise_for_status()` raises an exception for unsuccessful HTTP responses such as `404 Not Found` or `500 Internal Server Error`. This prevents later code from quietly treating an error page as a report.

The `Content-Type` header is an early clue about what the server returned. We expect `application/pdf`, although servers can occasionally provide incomplete or incorrect headers. For that reason, we also inspect the PDF *magic bytes*: genuine PDF files normally begin with `%PDF-`.

> **Common pitfall:** A URL ending in `.pdf` does not guarantee that the response is a PDF. A server might return an HTML login page or error message with a successful-looking filename.

In [4]:
content_type = response.headers.get("Content-Type", "")
print("Content-Type:", content_type)
print("Content-Length header:", response.headers.get("Content-Length"))
print("First five bytes:", response.content[:5])

if "pdf" not in content_type.lower():
    raise ValueError(f"Expected a PDF Content-Type, received {content_type!r}")
if not response.content.startswith(b"%PDF-"):
    raise ValueError("The response body does not have a PDF file signature.")

Content-Type: application/pdf
Content-Length header: 1709376
First five bytes: b'%PDF-'


We write the response body only when the local file is absent. This avoids overwriting an existing download unnecessarily. Mode `"wb"` means **write binary**: `"w"` expects text and can corrupt or reject binary PDF bytes.

In [5]:
if pdf_path.exists():
    print(f"Using existing file: {pdf_path}")
else:
    with pdf_path.open("wb") as file:
        file.write(response.content)
    print(f"Downloaded: {pdf_path}")

print("Exists:", pdf_path.exists())
print("Local size (bytes):", pdf_path.stat().st_size)

Using existing file: data/raw/kinzie_industrial_corridor_2019.pdf
Exists: True
Local size (bytes): 1709376


### Check your understanding

1. What does `requests.get()` return?
2. Why do we use `response.content` instead of `response.text`?
3. What does `raise_for_status()` do?
4. Why is the PDF written using `"wb"`?

## Part 2 — Inspect the PDF

`pdfplumber` can read text and geometric information from text-based PDFs. It is not performing OCR here. First, we open the document and count its pages.

In [6]:
with pdfplumber.open(pdf_path) as pdf:
    page_count = len(pdf.pages)

print("Number of pages:", page_count)

Number of pages: 39


Python uses **zero-based indexing**, so `pdf.pages[0]` is the first page. A PDF viewer normally labels that same page as page 1. To avoid hard-coding a fragile page index, we search page text for the section title. This also illustrates how to locate relevant content programmatically.

In [7]:
search_phrase = "Provide an analysis of the special tax allocation fund"

with pdfplumber.open(pdf_path) as pdf:
    matching_page_indexes = [
        index
        for index, page in enumerate(pdf.pages)
        if search_phrase.lower() in (page.extract_text() or "").lower()
    ]

financial_page_index = matching_page_indexes[0]
print("Python page index:", financial_page_index)
print("PDF viewer page number:", financial_page_index + 1)

Python page index: 5
PDF viewer page number: 6


Now extract the page as text. Printing only the first 2,500 characters keeps the notebook readable while preserving the important financial section.

In [8]:
with pdfplumber.open(pdf_path) as pdf:
    financial_page = pdf.pages[financial_page_index]
    financial_text = financial_page.extract_text()

print(financial_text[:2500])

SECTION 3.1 - (65 ILCS 5/11-74.4-5 (d)(5)(a)(b)(d)) and (65 ILCS 5/11-74.6-22 (d) (5)(a)(b)(d))
Provide an analysis of the special tax allocation fund.
FY 2019
TIF NAME: Kinzie Industrial Corridor Redevelopment Project Area
Special Tax Allocation Fund Balance at Beginning of Reporting Period: $ 95,678,423
Cumulative
Revenue/Cash Totals of
SOURCE of Revenue/Cash Receipts: Receipts for Revenue/Cash
Current Receipts for
Reporting Year life of TIF % of Total
Property Tax Increment $ 41,093,833 $ 361,813,998 97%
State Sales Tax Increment $ - $ - 0%
Local Sales Tax Increment $ - $ - 0%
State Utility Tax Increment $ - $ - 0%
Local Utility Tax Increment $ - $ - 0%
Interest $ 5,123,924 $ 9,927,053 3%
Land/Building Sale Proceeds $ - $ - 0%
Bond Proceeds $ - $ - 0%
Transfers from Municipal Sources $ - $ - 0%
Private Sources $ - $ - 0%
Other (identify source __________________; if multiple other $ - $ 1,136,624 0%
sources, attach schedule)
All Amount Deposited in Special Tax Allocation Fund $ 46,2

## Part 3 — Extract a financial table

PDFs do not necessarily contain a true table object. `pdfplumber` infers rows and columns from the positions of words and lines. `extract_tables()` returns a list of tables, and each table is a nested list: an outer list of rows, with an inner list of cell values for each row.

In [9]:
with pdfplumber.open(pdf_path) as pdf:
    raw_tables = pdf.pages[financial_page_index].extract_tables()

print("Tables found:", len(raw_tables))
raw_revenue_table = raw_tables[0]
raw_revenue_table[:5]

Tables found: 3


[['SOURCE of Revenue/Cash Receipts:',
  'Revenue/Cash\nReceipts for\nCurrent\nReporting Year',
  'Cumulative\nTotals of\nRevenue/Cash\nReceipts for\nlife of TIF',
  '% of Total'],
 ['Property Tax Increment', '$ 41,093,833', '$ 361,813,998', '97%'],
 ['State Sales Tax Increment', '$ -', '$ -', '0%'],
 ['Local Sales Tax Increment', '$ -', '$ -', '0%'],
 ['State Utility Tax Increment', '$ -', '$ -', '0%']]

The first row looks like a header, so we use it for column names and the remaining rows as data. We are not assuming the result is perfect—we inspect it next.

In [10]:
revenue_df = pd.DataFrame(
    raw_revenue_table[1:],
    columns=raw_revenue_table[0],
)
revenue_df.head()

,SOURCE of Revenue/Cash Receipts:,Revenue/Cash\nReceipts for\nCurrent\nReporting Year,Cumulative\nTotals of\nRevenue/Cash\nReceipts for\nlife of TIF,% of Total
0,Property Tax Increment,"$ 41,093,833","$ 361,813,998",97%
1,State Sales Tax Increment,$ -,$ -,0%
2,Local Sales Tax Increment,$ -,$ -,0%
3,State Utility Tax Increment,$ -,$ -,0%
4,Local Utility Tax Increment,$ -,$ -,0%


Use several views when checking extracted data. `head()` previews records, `shape` reports rows and columns, `columns` reveals awkward labels, and `info()` reports types and missing values.

In [11]:
print("Shape:", revenue_df.shape)
print("Columns:", revenue_df.columns.tolist())
revenue_df.info()

Shape: (11, 4)
Columns: ['SOURCE of Revenue/Cash Receipts:', 'Revenue/Cash\nReceipts for\nCurrent\nReporting Year', 'Cumulative\nTotals of\nRevenue/Cash\nReceipts for\nlife of TIF', '% of Total']
<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 4 columns):
 #   Column                                                      Non-Null Count  Dtype
---  ------                                                      --------------  -----
 0   SOURCE of Revenue/Cash Receipts:                            11 non-null     str  
 1   Revenue/Cash
Receipts for
Current
Reporting Year            11 non-null     str  
 2   Cumulative
Totals of
Revenue/Cash
Receipts for
life of TIF  11 non-null     str  
 3   % of Total                                                  11 non-null     str  
dtypes: str(4)
memory usage: 484.0 bytes


PDF extraction commonly produces merged cells, blank rows, repeated headers, multi-line labels, separated digits, missing values, and tables split across pages. Here, line breaks remain inside column names and one logical financial section was split into three inferred tables.

> **Common pitfall:** A table that looks perfectly aligned on screen may only be individually positioned words. The visual layout does not guarantee an underlying row-and-column structure.

## Part 4 — Extract the variables needed for analysis

We will create one tidy TIF-year record. Some values are in the inferred revenue table; summary fields such as expenses and ending balance occur in nearby page text; administrative and financing costs are in the multi-page expenditure table.

We retain the raw strings at this stage. Separating **extraction** from **cleaning** makes errors easier to diagnose.

In [12]:
current_year_column = revenue_df.columns[1]
cumulative_column = revenue_df.columns[2]

def table_value(table, label, column):
    matches = table[table.iloc[:, 0].str.contains(label, case=False, na=False, regex=False)]
    if len(matches) != 1:
        raise ValueError(f"Expected one row containing {label!r}; found {len(matches)}")
    return matches.iloc[0][column]


raw_values = {
    "property_tax_extraction": table_value(
        revenue_df, "Property Tax Increment", current_year_column
    ),
    "cumulative_pte": table_value(
        revenue_df, "Property Tax Increment", cumulative_column
    ),
    "transfers_in": table_value(
        revenue_df, "Transfers from Municipal Sources", current_year_column
    ),
    "cumulative_transfers_in": table_value(
        revenue_df, "Transfers from Municipal Sources", cumulative_column
    ),
}
raw_values

{'property_tax_extraction': '$ 41,093,833',
 'cumulative_pte': '$ 361,813,998',
 'transfers_in': '$ -',
 'cumulative_transfers_in': '$ -'}

For summary lines, a small helper finds a line by its descriptive label and returns the text after the first dollar sign. This is intentionally simple and readable; it also fails loudly if the label is absent or duplicated.

In [13]:
def amount_after_label(text, label):
    matching_lines = [
        line for line in text.splitlines()
        if line.strip().lower().startswith(label.lower())
    ]
    if len(matching_lines) != 1:
        raise ValueError(f"Expected one line beginning {label!r}; found {len(matching_lines)}")
    line = matching_lines[0]
    if "$" not in line:
        raise ValueError(f"No dollar amount found on line: {line}")
    return "$" + line.split("$", maxsplit=1)[1].strip()


raw_values.update({
    "expenses": amount_after_label(
        financial_text, "Total Expenditures/Cash Disbursements (Carried forward"
    ),
    "fund_balance_end": amount_after_label(
        financial_text, "FUND BALANCE, END OF REPORTING PERIOD"
    ),
    "transfers_out": amount_after_label(
        financial_text, "Transfers to Municipal Sources"
    ),
    "distribution": amount_after_label(
        financial_text, "Distribution of Surplus"
    ),
})
raw_values

{'property_tax_extraction': '$ 41,093,833',
 'cumulative_pte': '$ 361,813,998',
 'transfers_in': '$ -',
 'cumulative_transfers_in': '$ -',
 'expenses': '$17,400,749',
 'fund_balance_end': '$124,495,431',
 'transfers_out': '$-',
 'distribution': '$-'}

The expenditure categories span report pages 8–10 (Python indexes 7–9). We inspect all inferred rows and search forward from a category label to its value in the final column. This handles the blank rows inserted by the page layout.

In [14]:
with pdfplumber.open(pdf_path) as pdf:
    expenditure_tables = [
        table
        for page_index in range(7, 10)
        for table in pdf.pages[page_index].extract_tables()
    ]

def category_value(tables, category_label):
    for table in tables:
        for row_index, row in enumerate(table):
            first_cell = (row[0] or "").lower()
            if category_label.lower() in first_cell:
                for later_row in table[row_index + 1:]:
                    if later_row[0]:  # The next category has begun.
                        break
                    if later_row[-1] and "$" in later_row[-1]:
                        return later_row[-1]
    raise ValueError(f"Could not find a value for {category_label!r}")


raw_values["administration_costs"] = category_value(
    expenditure_tables, "Annual administrative cost"
)
raw_values["finance_costs"] = category_value(
    expenditure_tables, "Financing costs"
)
raw_values

{'property_tax_extraction': '$ 41,093,833',
 'cumulative_pte': '$ 361,813,998',
 'transfers_in': '$ -',
 'cumulative_transfers_in': '$ -',
 'expenses': '$17,400,749',
 'fund_balance_end': '$124,495,431',
 'transfers_out': '$-',
 'distribution': '$-',
 'administration_costs': '$ -',
 'finance_costs': '$ -'}

The 2019 PDF does not state a Chicago TIF number or name a depository bank. Missing source information should remain missing—not be inferred from unrelated text. We will represent both as `pd.NA` and document the limitation.

The 2010 Kinzie PDF was not available at a reliably accessible public URL during lesson preparation. Therefore the executable demonstration uses the official 2019 report. The same variable-oriented workflow can be adapted to 2010 when that original file is obtained, but page locations and labels must be rechecked.

## Part 5 — Clean currency values

Financial values arrive as strings. A reusable cleaner removes dollar signs, spaces, and commas; treats a dash or empty string as zero/missing as appropriate; and recognizes parentheses as an accounting-style negative sign. We return integers because these report values are whole dollars.

In [15]:
def clean_currency(value):
    if value is None or pd.isna(value):
        return None

    cleaned = str(value).strip().replace("\n", "")
    if cleaned == "":
        return None
    if cleaned in {"-", "$-", "$ -"}:
        return 0

    is_negative = cleaned.startswith("(") and cleaned.endswith(")")
    cleaned = cleaned.strip("()")
    cleaned = cleaned.replace("$", "").replace(",", "").replace(" ", "")

    number = int(cleaned)
    return -number if is_negative else number

Small examples serve as readable tests and show the intended behavior before we apply the function to report values.

In [16]:
currency_tests = ["$17,458,853", "$0", "(1,250)", "$ 280,419", "", None, "$ -"]
[(value, clean_currency(value)) for value in currency_tests]

[('$17,458,853', 17458853),
 ('$0', 0),
 ('(1,250)', -1250),
 ('$ 280,419', 280419),
 ('', None),
 (None, None),
 ('$ -', 0)]

Now apply the cleaner to each extracted financial field. Notice that the inputs come from `raw_values`, not from the historical reference figures.

In [17]:
clean_values = {
    variable: clean_currency(value)
    for variable, value in raw_values.items()
}
clean_values

{'property_tax_extraction': 41093833,
 'cumulative_pte': 361813998,
 'transfers_in': 0,
 'cumulative_transfers_in': 0,
 'expenses': 17400749,
 'fund_balance_end': 124495431,
 'transfers_out': 0,
 'distribution': 0,
 'administration_costs': 0,
 'finance_costs': 0}

## Part 6 — Build a tidy record

In a **tidy** dataset, each row is one observation and each column is one variable. Here, one observation is one TIF district in one reporting year. This structure makes it easy to filter years, compare districts, group records, and append future reports.

In [18]:
record = {
    "tif_number": pd.NA,  # Not printed in this PDF.
    "tif_name": "Kinzie Industrial Corridor",
    "year": 2019,
    **clean_values,
    "bank_names": pd.NA,  # No named depository bank appears in this PDF.
}

result = pd.DataFrame([record])
result

,tif_number,tif_name,year,property_tax_extraction,cumulative_pte,transfers_in,cumulative_transfers_in,expenses,fund_balance_end,transfers_out,distribution,administration_costs,finance_costs,bank_names
0,<NA>,Kinzie Industrial Corridor,2019,41093833,361813998,0,0,17400749,124495431,0,0,0,0,<NA>


## Part 7 — Validate the extraction

PDF extraction should never be trusted blindly. The reference values below were manually transcribed from the displayed FY 2019 annual-report page and expenditure schedules. They are independent checks, not inputs to `record`.

Validation is part of data acquisition: a parsing error can produce a plausible-looking but incorrect number.

In [19]:
reference_2019 = {
    "property_tax_extraction": 41_093_833,
    "cumulative_pte": 361_813_998,
    "transfers_in": 0,
    "cumulative_transfers_in": 0,
    "expenses": 17_400_749,
    "fund_balance_end": 124_495_431,
    "transfers_out": 0,
    "distribution": 0,
    "administration_costs": 0,
    "finance_costs": 0,
}

validation = pd.DataFrame({
    "variable": reference_2019.keys(),
    "extracted_value": [record[name] for name in reference_2019],
    "reference_value": reference_2019.values(),
})
validation["match"] = validation["extracted_value"] == validation["reference_value"]
validation

,variable,extracted_value,reference_value,match
0,property_tax_extraction,41093833,41093833,True
1,cumulative_pte,361813998,361813998,True
2,transfers_in,0,0,True
3,cumulative_transfers_in,0,0,True
4,expenses,17400749,17400749,True
5,fund_balance_end,124495431,124495431,True
6,transfers_out,0,0,True
7,distribution,0,0,True
8,administration_costs,0,0,True
9,finance_costs,0,0,True


An assertion makes a mismatch stop the notebook instead of passing silently.

In [20]:
assert validation["match"].all(), "At least one extracted value failed validation."
print(f"Validated {validation['match'].sum()} of {len(validation)} financial fields.")

Validated 10 of 10 financial fields.


### Historical 2010 Kinzie portal reference

The following known portal values are useful targets if the original 2010 Kinzie report becomes available. They are shown only as a reference table. We deliberately do **not** compare them with the 2019 extraction because the years differ, and we do not use them to populate the result.

In [21]:
kinzie_2010_portal_reference = pd.DataFrame([{
    "year": 2010,
    "property_tax_extraction": 17_458_853,
    "cumulative_pte": 124_608_186,
    "transfers_in": 0,
    "cumulative_transfers_in": 0,
    "expenses": 14_008_782,
    "fund_balance_end": 50_393_394,
    "transfers_out": 2_974_054,
    "distribution": 0,
    "administration_costs": 280_419,
    "finance_costs": 0,
}])
kinzie_2010_portal_reference

,year,property_tax_extraction,cumulative_pte,transfers_in,cumulative_transfers_in,expenses,fund_balance_end,transfers_out,distribution,administration_costs,finance_costs
0,2010,17458853,124608186,0,0,14008782,50393394,2974054,0,280419,0


## Part 8 — Export to CSV

`index=False` prevents pandas from adding the DataFrame's row index as an extra CSV column. Reading the file back confirms that it exists, parses successfully, and has the expected shape and fields.

### Data Provenance

- **Data publisher:** City of Chicago
- **Source:** TIF District Annual Reports
- **Executable report:** [FY 2019 Kinzie Industrial Corridor report](https://files.illinoiscomptroller.gov/LocGovTIF/FY2019/01662030/19TIF01662030Kinzie_InCorridor_1.pdf)
- **Original format:** PDF
- **Community project:** MCDC TIF Illumination and Analysis
- **Community partner:** The CivicLab
- **Extraction method used here:** Python (`requests`, `pdfplumber`, and `pandas`)
- **Output format:** CSV

Recording provenance allows anyone using the CSV to trace its values back to the original public report and assess how the data were produced.

In [22]:
result.to_csv(csv_path, index=False)
print(f"Wrote {csv_path} ({csv_path.stat().st_size} bytes)")

round_trip = pd.read_csv(csv_path)
round_trip

Wrote data/processed/kinzie_2019_tif.csv (284 bytes)


,tif_number,tif_name,year,property_tax_extraction,cumulative_pte,transfers_in,cumulative_transfers_in,expenses,fund_balance_end,transfers_out,distribution,administration_costs,finance_costs,bank_names
0,NaN,Kinzie Industrial Corridor,2019,41093833,361813998,0,0,17400749,124495431,0,0,0,0,NaN


We can also verify that the round-tripped financial columns retained their values.

In [23]:
financial_columns = list(reference_2019)
assert round_trip.loc[0, financial_columns].to_dict() == {
    name: reference_2019[name] for name in financial_columns
}
print("CSV round-trip validation passed.")

CSV round-trip validation passed.


## Part 9 — From one PDF to many PDFs

Once the steps work separately, we can package repeated operations into functions. `download_pdf` validates the response and reuses an existing file. It does not silently suppress network or file errors.

### MCDC Connection: From Public Information to Usable Data

Converting reports into a structured dataset makes it possible to compare TIF districts, study change over time, create visualizations, and investigate questions that would be difficult to address by manually reading PDFs one at a time.

In [24]:
def download_pdf(url, output_path):
    output_path = Path(output_path)
    if output_path.exists():
        print(f"Already downloaded: {output_path}")
        return output_path

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    content_type = response.headers.get("Content-Type", "")
    if "pdf" not in content_type.lower() or not response.content.startswith(b"%PDF-"):
        raise ValueError("The downloaded resource does not appear to be a PDF.")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("wb") as file:
        file.write(response.content)
    return output_path

The extractor below captures the central idea without pretending every Chicago report has the same layout. It locates Section 3.1 and returns its page text plus its main table. A production extractor would then apply the carefully validated field rules developed above.

In [25]:
def extract_tif_report(pdf_path):
    pdf_path = Path(pdf_path)
    marker = "Provide an analysis of the special tax allocation fund"

    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            if marker.lower() in text.lower():
                tables = page.extract_tables()
                if not tables:
                    raise ValueError(f"No table found on report page {page_number}.")
                return {
                    "page_number": page_number,
                    "text": text,
                    "revenue_table": tables[0],
                }

    raise ValueError("Could not locate Section 3.1 in the PDF.")


report_section = extract_tif_report(pdf_path)
print("Located Section 3.1 on PDF page", report_section["page_number"])
print("Revenue-table rows:", len(report_section["revenue_table"]))

Located Section 3.1 on PDF page 6
Revenue-table rows: 12


Eventually, metadata could drive a loop:

```python
records = []
for report in reports:
    pdf_path = download_pdf(report["url"], report["output_path"])
    extracted = extract_tif_report(pdf_path)
    records.append(extracted)

combined = pd.DataFrame(records)
```

Do not automate hundreds of reports until a small, varied sample validates correctly. Layouts may change across years; reports or pages may be missing; downloads may fail; TIF names may change; page numbers can shift; and table geometry may differ. Good pipelines log such failures for review rather than quietly creating bad data.

## Part 10 — Exercises

### Exercise 1: Response metadata

Inspect `response.headers`. Report the `Content-Type` and either the `Content-Length` header or `len(response.content)`. Why might those two size values differ?

### Exercise 2: Locate another table

Extract text from another page. Write a loop that prints page numbers containing the phrase `ITEMIZED LIST OF ALL EXPENDITURES`. Remember to translate Python indexes to viewer page numbers.

### Exercise 3: More currency cases

Add tests showing that `clean_currency()` processes `(2,500)` and `$ (2,500)` correctly. If the second case fails, revise the function while preserving all earlier behavior.

### Exercise 4: Add a financial variable

Extract current-year `Interest` revenue from `revenue_df`, clean it, and add it to a copy of `record`. Validate it against the original report page.

### Exercise 5: Construct a filename

Create a safe filename that includes the TIF name and year, such as `kinzie_industrial_corridor_2019.csv`, and save the result in `data/processed/`.

### Challenge: Combine two reports

Apply the workflow to a second accessible Chicago TIF annual report. Keep one row per TIF-year and combine the two DataFrames using `pd.concat([...], ignore_index=True)`. Record any layout changes you encounter.

## Part 11 — Reflection and discussion

1. Why is extracting a PDF different from reading a CSV file?
2. Why might a PDF table look correct to a person but be difficult for Python to parse?
3. Why is validation especially important with semi-structured data?
4. What could go wrong if a government agency changes the report format next year?
5. When would OCR be necessary? What extra errors might OCR introduce?
6. When might it be better to look for an API or existing structured dataset instead of parsing PDFs?

> **Why this matters:** Reproducibility is not merely rerunning code. It also requires stable sources, explicit assumptions, visible missingness, and evidence that extracted values agree with the authoritative document.

## From Extraction to Application

After many annual PDF reports have been converted into structured data, the result can support downstream tools such as interactive maps and dashboards. The original project developed the [Chicago Top TIFs application](https://wtfineberg.users.earthengine.app/view/toptifs) in Google Earth Engine as one example. This lesson does not cover Earth Engine; the application simply illustrates why a careful PDF-to-CSV workflow matters in a larger data science project.

## Reflection: Data for Community Use

1. The TIF reports were already public. What additional value is created by converting their information from PDFs into a structured CSV?
2. Why might a community organization prefer a reproducible extraction process instead of manually copying values?
3. What errors could occur during automated PDF extraction, and why would they matter if community organizations, journalists, researchers, or policymakers used the resulting dataset?
4. If Chicago changed its PDF report format next year, how could that affect the extraction pipeline?
5. What documentation should accompany a dataset created from public PDF reports so another person can verify where the numbers came from?

## Source notes and limitations

- Executable source: Illinois Comptroller, *FY 2019 Annual Tax Increment Finance Report: Kinzie Industrial Corridor Redevelopment Project Area*.
- The file is text-based; `extract_text()` returns readable text, so OCR is unnecessary.
- The PDF reports neither a TIF number nor named depository banks; these fields are left missing.
- The exact 2010 Kinzie PDF could not be located at a stable, programmatically accessible public URL during preparation. Its supplied portal figures are included only as historical validation targets for future work with that file.
- PDF parsers and agency websites change. If extraction stops matching, inspect the original page before modifying rules.

## Acknowledgment

This lesson builds on the [Tax Increment Financing (TIF) Illumination and Analysis project](https://sites.northwestern.edu/mcdc/tax-increment-financing-illumination-and-analysis/) conducted through the Metropolitan Chicago Data Science Corps (MCDC) in collaboration with The CivicLab. The [original project repository](https://github.com/philipayates/chicago2022TIF) contains data-processing and visualization resources developed to examine Chicago TIF finances.